# Solution: ARIMA Order Selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
DATA_PATH = "../data/electricity_demand.csv"


In [ ]:
def load_demand(path):
    df = pd.read_csv(path, parse_dates=["date"], index_col="date").asfreq("MS")
    return df.loc[:"2023-12-01", "demand_mwh"]

train = load_demand(DATA_PATH)
print(f"Train: {len(train)} months")


In [ ]:
def grid_search_arima(train, p_vals, d_vals, q_vals):
    results = []
    for p, d, q in product(p_vals, d_vals, q_vals):
        try:
            model = SARIMAX(train, order=(p, d, q))
            fitted = model.fit(disp=False)
            results.append({"order": (p, d, q), "aic": fitted.aic, "bic": fitted.bic, "fitted": fitted})
        except Exception:
            continue
    return sorted(results, key=lambda x: x["aic"])[0]

best = grid_search_arima(train, p_vals=[0, 1, 2], d_vals=[1], q_vals=[0, 1, 2])
print(f"Best order: {best['order']}  AIC: {best['aic']:.1f}")


In [ ]:
def diagnose_winner(fitted):
    resid = fitted.resid
    lb = acorr_ljungbox(resid, lags=[12], return_df=True)
    _, norm_p = stats.normaltest(resid.dropna())
    return {
        "order": fitted.model.order,
        "aic": fitted.aic,
        "ljung_box_p": lb["lb_pvalue"].iloc[0],
        "normality_p": norm_p
    }

diag = diagnose_winner(best["fitted"])
for k, v in diag.items():
    print(f"{k}: {v}")


In [ ]:
def plot_winner_acf(fitted, lags=24):
    fig, ax = plt.subplots(figsize=(14, 4))
    plot_acf(fitted.resid, ax=ax, lags=lags, color=UB["Brand Blue"])
    ax.set_title(f"Residual ACF for {fitted.model.order}", fontsize=18, fontweight="bold")
    ax.set_ylabel("Autocorrelation", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    plt.tight_layout()
    plt.show()

plot_winner_acf(best["fitted"])
